# **Equivalence and roles**

This notebook provides exercises related to:
- Structural equivalence
- Automorphic equivalence
- Regular equivalence

These concepts are central to role analysis in social and technological networks.

## Setup

In [ ]:
import networkx as nx
from networkx.drawing.nx_pydot import graphviz_layout
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pydot
from collections import Counter, defaultdict

print('NetworkX version:', nx.__version__)


## **Structural equivalence**

Two nodes are structurally equivalent if they have *identical connections to identical others*.
- In an undirected graph: they must have exactly the same neighbor set.
- In a directed graph: they must have the same set of in-neighbors and the same set of out-neighbors.

Structural equivalence is very strict:
- It assumes that roles are defined by *exactly the same local pattern* of ties.
- In real, noisy networks, exact structural equivalence is rare.
- Therefore, we often use *similarity measures* between rows of the adjacency matrix (e.g. Jaccard similarity, cosine similarity, Pearson correlation) to capture approximate structural equivalence.

### Exercise
1. Build a small undirected graph with a few intentionally structurally equivalent nodes.
2. Visualize the network.
3. Compute *exact structural equivalence classes* (based on identical neighbor sets).
4. Compute a *similarity matrices* between neighbor sets.
5. Compare exact and approximate equivalence and reflect: which pairs are *almost* structurally equivalent?

In [ ]:
# Example graph
G = nx.Graph()
G.add_edges_from([
    ('A', 'B'), ('A', 'C'), ('A', 'D'),
    ('B', 'E'), ('B', 'F'), ('C', 'G'),
    ('D', 'H'), ('D', 'I')
])


pos = graphviz_layout(G, prog="twopi")
plt.figure(figsize=(6, 5))
nx.draw(G, pos, with_labels=True)
plt.title('Example graph for structural equivalence')
plt.show()

In [ ]:
# Exact structural equivalence classes

def neighbors_set(G, node):
    return set(G.neighbors(node))

nodes = list(G.nodes())
neigh = {n: neighbors_set(G, n) for n in nodes}

classes = []
unassigned = set(nodes)

while unassigned:
    n = unassigned.pop()
    cls = {n}
    for m in list(unassigned):
        if neigh[m] == neigh[n]:
            cls.add(m)
            unassigned.remove(m)
    classes.append(cls)

print('Structural equivalence classes:')
for c in classes:
    print(c)

In [ ]:
# Jaccard similarity between neighbor sets
def jaccard_similarity(a, b):
    inter = len(a & b)
    union = len(a | b)
    if union == 0:
        return 0.0
    return inter / union

# Cosine similarity between neighbor sets (binary vectors)
def cosine_similarity(a, b):
    inter = len(a & b)
    na = len(a)
    nb = len(b)
    if na == 0 or nb == 0:
        return 0.0
    return inter / math.sqrt(na * nb)

# Jaccard matrix
jac_mat = pd.DataFrame(index=nodes, columns=nodes, dtype=float)
for i in nodes:
    for j in nodes:
        if i == j:
            jac_mat.loc[i, j] = 1.0
        else:
            jac_mat.loc[i, j] = jaccard_similarity(neigh[i], neigh[j])

# Cosine matrix
cos_mat = pd.DataFrame(index=nodes, columns=nodes, dtype=float)
for i in nodes:
    for j in nodes:
        if i == j:
            cos_mat.loc[i, j] = 1.0
        else:
            cos_mat.loc[i, j] = cosine_similarity(neigh[i], neigh[j])

# Plot Jaccard correlogram
plt.figure(figsize=(8, 6))
sns.heatmap(jac_mat.astype(float), annot=True, vmin=0, vmax=1,
            cmap="coolwarm", square=True)
plt.title("Jaccard similarity between neighbor sets")
plt.show()

# Plot Cosine correlogram
plt.figure(figsize=(8, 6))
sns.heatmap(cos_mat.astype(float), annot=True, vmin=0, vmax=1,
            cmap="coolwarm", square=True)
plt.title("Cosine similarity between neighbor sets")
plt.show()


**Discussion:**
- Which node pairs are exactly structurally equivalent?
- Which pairs have *high similarity* but are not exactly equivalent?
- How could we use a threshold (e.g. Jaccard > 0.8) to define *approximate* structural equivalence classes?

## **Automorphic equivalence**

Two nodes are automorphically equivalent if there exists a *graph automorphism* (a structure-preserving permutation of nodes) that maps one node to the other.

- An automorphism is a bijection `f: V → V` such that `(u, v)` is an edge if and only if `(f(u), f(v))` is an edge.
- Automorphic equivalence is less strict than structural equivalence: nodes do *not* need to have exactly the same neighbors, only the same *position* in a repeated or symmetric pattern.
- This captures the idea of “parallel substructures” (e.g. different branches of a franchise, or different classes in a school).

### Exercise
Consider a toy organizational network: one headquarters, two managers, and several workers.
1. Construct the network.
2. Use NetworkX's `GraphMatcher` to enumerate automorphisms.
3. Build automorphic equivalence classes: two nodes are equivalent if some automorphism maps one to the other.
4. Interpret the classes as *roles* (center, manager, worker).


In [ ]:
# Toy organization: fast-food chain structure
H = nx.Graph()

# Nodes with role attribute
H.add_node('C', role='center')  # headquarters
H.add_node('M1', role='manager')
H.add_node('M2', role='manager')
for w in ['W1a', 'W1b', 'W2a', 'W2b']:
    H.add_node(w, role='worker')

# Edges: center to managers
H.add_edges_from([
    ('C', 'M1'), ('C', 'M2'),
])

# Edges: managers to workers
H.add_edges_from([
    ('M1', 'W1a'), ('M1', 'W1b'),
    ('M2', 'W2a'), ('M2', 'W2b'),
])

pos = nx.spring_layout(H, seed=5)
roles = nx.get_node_attributes(H, 'role')
color_map = {'center': 'tab:red', 'manager': 'tab:blue', 'worker': 'tab:green'}
colors = [color_map[roles[n]] for n in H.nodes()]

plt.figure(figsize=(6, 5))
nx.draw(H, pos, with_labels=True, node_color=colors)
plt.title('Organizational toy network for automorphic equivalence')
plt.show()

In [ ]:
# Enumerate automorphisms and derive automorphic equivalence classes
from networkx.algorithms import isomorphism

GM = isomorphism.GraphMatcher(H, H)
automorphisms = list(GM.isomorphisms_iter())
print('Number of automorphisms:', len(automorphisms))
print('Example automorphism:', automorphisms[0])

In [ ]:
def are_automorphically_equiv(u, v, automorphisms):
    """Return True if there exists an automorphism mapping u to v."""
    for iso in automorphisms:
        if iso.get(u) == v:
            return True
    return False

nodes_H = list(H.nodes())
assigned = set()
auto_classes = []

for u in nodes_H:
    if u in assigned:
        continue
    cls = {u}
    for v in nodes_H:
        if v != u and v not in assigned and are_automorphically_equiv(u, v, automorphisms):
            cls.add(v)
    auto_classes.append(cls)
    assigned |= cls

print('Automorphic equivalence classes:')
for c in auto_classes:
    print(c)

## **Regular equivalence**

Two nodes are regularly equivalent if they are related in similar ways to nodes that are themselves regularly equivalent.

Intuition:
- Regular equivalence aims to capture *roles* independent of specific identities.
- Example: in a family network, all parents are regularly equivalent to each other, and all children are regularly equivalent to each other, even if they do not all know each other directly.

Formally, regular equivalence is defined recursively and can be computed by specialized algorithms (e.g. REGE).

Here we will approximate it via an *iterative label refinement* process:
1. Assign initial labels to nodes (e.g. all unique, or based on simple attributes like degree).
2. In each iteration, update a node's label based on the multiset of labels of its neighbors.
3. Compress these profiles into new labels.
4. Repeat until labels stabilize or after a fixed number of iterations.

### Exercise
Build a simple directed Source -> Broker -> Target network and apply iterative label refinement to discover role-like classes.

1. *Initialize*  
   Give all nodes the same label (e.g. `R0`).

2. *Build profiles*  
   For each node:  
   - collect the set of labels of its in-neighbors  
   - collect the set of labels of its out-neighbors  
   - profile = (in-label-set, out-label-set)

3. *Relabel*  
   Nodes with identical profiles receive the same new label.

4. *Iterate*  
   Repeat steps 2 and 3 until labels stop changing.

5. *Output*  
   Final labels = approximate regular-equivalence classes (role groups).

In [ ]:
# Source–Broker–Target directed network
R = nx.DiGraph()

sources = ['S1', 'S2']
brokers = ['B1', 'B2', 'B3']
targets = ['T1', 'T2', 'T3', 'T4']

R.add_nodes_from(sources, true_role='source')
R.add_nodes_from(brokers, true_role='broker')
R.add_nodes_from(targets, true_role='target')

R.add_edges_from([
    ('S1', 'B1'), ('S1', 'B2'),
    ('S2', 'B2'), ('S2', 'B3'),
])
R.add_edges_from([
    ('B1', 'T1'), ('B1', 'T2'),
    ('B2', 'T2'), ('B2', 'T3'),
    ('B3', 'T3'), ('B3', 'T4'),
])

pos_R = nx.spring_layout(R, seed=3)
true_roles = nx.get_node_attributes(R, 'true_role')
color_map_R = {'source': 'tab:red', 'broker': 'tab:blue', 'target': 'tab:green'}
colors_R = [color_map_R[true_roles[n]] for n in R.nodes()]

plt.figure(figsize=(7, 5))
nx.draw(R, pos_R, with_labels=True, node_color=colors_R, arrows=True)
plt.title('Source–Broker–Target network')
plt.show()

In [ ]:
def update_labels_regular(G, labels):
    """
    One refinement step for approximate regular equivalence
    on a directed graph G.

    labels: dict node -> current class label (str)
    """
    profiles = {}

    for n in G.nodes():
        # set of types of nodes pointing to n
        in_types = frozenset(labels[u] for u in G.predecessors(n))
        # set of types of nodes n points to
        out_types = frozenset(labels[v] for v in G.successors(n))

        # profile ignores multiplicities: just which types appear
        profiles[n] = (in_types, out_types)

    # compress profiles into new compact labels
    profile_to_label = {}
    new_labels = {}
    for n, prof in profiles.items():
        if prof not in profile_to_label:
            profile_to_label[prof] = f'R{len(profile_to_label)}'
        new_labels[n] = profile_to_label[prof]

    return new_labels


def regular_equivalence_refinement(G, max_iter=10, verbose=False):
    """
    Iteratively refine labels to approximate regular equivalence.
    All nodes start in the same class.
    """
    # start with all nodes in one class
    labels = {n: 'R0' for n in G.nodes()}

    for it in range(max_iter):
        new_labels = update_labels_regular(G, labels)
        if verbose:
            print(f"Iteration {it+1}: {new_labels}")
        if new_labels == labels:
            break   # converged
        labels = new_labels

    return labels

final_labels_R = regular_equivalence_refinement(R, verbose=True)
print(final_labels_R)


## **Example - Zachary's Karate Club**

In NetworkX, `nx.karate_club_graph()` provides this network with a node attribute `'club'` that indicates which side of the split each member ultimately joined:
- `'Mr. Hi'` faction
- `'Officer'` faction

This network will be used to illustrate:
- *Structural equivalence*: very few exact pairs.
- *Automorphic equivalence*: some symmetric positions within factions.
- *Regular/role equivalence*: leaders, core supporters, peripheral members.

In [ ]:
# Load and visualize the Karate Club network
KC = nx.karate_club_graph()

print('Number of nodes:', KC.number_of_nodes())
print('Number of edges:', KC.number_of_edges())

club_attr = nx.get_node_attributes(KC, 'club')
factions = sorted(set(club_attr.values()))
print('Factions:', factions)

color_map_KC = {factions[0]: 'tab:blue', factions[1]: 'tab:orange'} if len(factions) == 2 else {}
colors_KC = [color_map_KC.get(club_attr[n], 'tab:gray') for n in KC.nodes()]

pos_KC = nx.spring_layout(KC, seed=42)
plt.figure(figsize=(7, 6))
nx.draw(KC, pos_KC, with_labels=True, node_color=colors_KC)
plt.title("Zachary's Karate Club – nodes colored by faction ('club' attribute)")
plt.show()

#### **Structural equivalence and similarity on the Karate Club**

**Goal:**
- Measure how similar nodes are in terms of their neighborhoods.
- See that exact structural equivalence is rare, but some nodes are *close* in terms of Jaccard similarity.


In [ ]:
import math
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

nodes_KC = list(KC.nodes())
neigh_KC = {n: set(KC.neighbors(n)) for n in nodes_KC}

# Jaccard similarity between neighbor sets
def jaccard_similarity(a, b):
    inter = len(a & b)
    union = len(a | b)
    if union == 0:
        return 0.0
    return inter / union

# Cosine similarity between neighbor sets (binary vectors)
def cosine_similarity(a, b):
    inter = len(a & b)
    na = len(a)
    nb = len(b)
    if na == 0 or nb == 0:
        return 0.0
    return inter / math.sqrt(na * nb)

# Jaccard matrix
jac_mat = pd.DataFrame(index=nodes_KC, columns=nodes_KC, dtype=float)
for i in nodes_KC:
    for j in nodes_KC:
        if i == j:
            jac_mat.loc[i, j] = 1.0
        else:
            jac_mat.loc[i, j] = jaccard_similarity(neigh_KC[i], neigh_KC[j])

# Cosine matrix
cos_mat = pd.DataFrame(index=nodes_KC, columns=nodes_KC, dtype=float)
for i in nodes_KC:
    for j in nodes_KC:
        if i == j:
            cos_mat.loc[i, j] = 1.0
        else:
            cos_mat.loc[i, j] = cosine_similarity(neigh_KC[i], neigh_KC[j])

# Plot Jaccard correlogram
plt.figure(figsize=(10, 10))
sns.heatmap(
    jac_mat.astype(float),
    annot=True,
    annot_kws={"size": 6},
    vmin=0, vmax=1,
    cmap="coolwarm",
    square=True
)
plt.title("Jaccard similarity")
plt.show()

# Plot Cosine correlogram
plt.figure(figsize=(10, 10))
sns.heatmap(
    cos_mat.astype(float),
    annot=True,
    annot_kws={"size": 6},
    vmin=0, vmax=1,
    cmap="coolwarm",
    square=True
)
plt.title("Cosine similarity")
plt.show()


In [ ]:
# Find pairs with high cosine similarity (excluding self-pairs)
threshold = 0.7
high_sim_pairs = []
for i in nodes_KC:
    for j in nodes_KC:
        if i < j and cos_mat.loc[i, j] >= threshold:
            high_sim_pairs.append((i, j, cos_mat.loc[i, j]))

high_sim_pairs[:20]  # show first 20 pairs